Evaluate inference of fresh subjects who don't have ground truth segmentations

In [1]:
%matplotlib widget
import matplotlib.pyplot as plt

from scripts import lesion_diagnostics
import os
from pathlib import Path
import pandas as pd
import numpy as np
import nibabel as nib
import json
import pickle as pkl
from collections import defaultdict

import scripts.compile_run_metrics
import helpers.utils
import scripts.inference
from IPython.display import clear_output

from reload_recursive import reload_recursive
from tqdm.notebook import tqdm

from helpers.parallel import BetterPool

In [2]:
reload_recursive(scripts.compile_run_metrics)
#TODO Thi
from scripts.compile_run_metrics import EXPERIMENT_KEYS

reload_recursive(scripts.inference)
from scripts.inference import uncrop_predictions

import analysis.image
reload_recursive(analysis.image)
from analysis.image import lesion_analysis as lesion_dx

reload_recursive(helpers.utils)
from helpers.utils import dice_score, get_prl_indices

import helpers.shell_interface
reload_recursive(helpers.shell_interface)
from helpers.shell_interface import open_itksnap_workspace_cmd

import core.dataset
reload_recursive(core.dataset)
from core.dataset import Dataset

import core.experiment
reload_recursive(core.experiment)
from core.experiment import Experiment
from loguru import logger

import scripts
reload_recursive(scripts)
# from scripts import lesion_diagnostics as lesion_dx
from scripts.compile_run_metrics import load_or_cache_run

In [3]:
def get_infer_path(dataset, test_case, experiment_name) -> Path:
    case_dir = dataset.lesion_dir(test_case)
    matches = list(case_dir.glob(f"*{experiment_name.replace('/','_')}.nii.gz"))
    if len(matches) > 1:
        logger.warning(f"Found more than 1 case: {','.join(matches)}, returning the first")
    return matches[0]
    
def find_inference(search_path, experiment_name) -> Path:
    matches = list(search_path.glob(f"*{experiment_name.replace('/','_')}.nii.gz"))
    if len(matches) > 1:
        logger.warning(f"Found more than 1 case: {','.join(matches)}, returning the first")
    return matches[0]



In [4]:
import math
import traceback

def analyze_prl_case(prl_case: dict, subject_dir, bbox_suffix):
    subid = prl_case['subid']
    index = prl_case['lesion_index']
    data_root = subject_dir.parent

    lesion_index_path = subject_dir / "lstai_lesion_index.nii.gz"
    lesion_index = nib.load(str(lesion_index_path)).get_fdata().astype(np.int32)

    # Parse bounding boxes
    bbox_file = subject_dir / f"lstai_bounding_boxes{bbox_suffix}.txt"
    bounding_boxes = lesion_diagnostics._parse_bounding_boxes(bbox_file)

    try:
        assert bounding_boxes[index-1][0] == index
    except AssertionError:
        print(index, bounding_boxes[index-1][0])
    coords = bounding_boxes[index-1][1]

    # Load inference output for this ROI
    infer_path = data_root / prl_case["inference"]
    if not infer_path.exists():
        print(f"Inference output not found: {infer_path}")
        return None
    label_paths = [infer_path]
    
    label_keys = ["infer"]
    lesion_stats = {"lesion_index": index, **{f"rim_voxels_{k}": None for k in label_keys}}
    lesion_data = {"subid": subid, "lesion_index": index}
    for id, lab in zip(label_keys, label_paths):
        lab_nifti = nib.load(str(lab))
        lab_data = lab_nifti.get_fdata().astype(np.uint8)
        voxel_size = lab_nifti.header.get_zooms()[:3]
        voxel_volume = math.prod(voxel_size)

        lesion_data[f"label_{id}"] = lab_data

        # Crop lesion index with the same bounding box
        index_crop = lesion_diagnostics._crop_from_volume(lesion_index, coords)
        lesion_data[f"index_crop_{id}"] = index_crop

        try:
            # any iron detection: rim voxels that overlap the central lesion's footprint
            has_iron = np.any((index_crop == index) & (lab_data == 2))
            if id == "infer":
                lesion_stats['has_iron_infer'] = has_iron
            # ---Process Rim---
            # get the rim for the center lesion
            rim = lesion_dx._get_lesion_rim(index_crop, lab_data, index)
            rim_count = int(rim.sum())
            rim_sphere_radius = lesion_dx.rim_enclosing_sphere_radius(rim, voxel_size)

            # get convex hull
            hull = lesion_dx.get_convex_hull(rim, voxel_sizes=voxel_size)
            
            lesion_data.update({
                f"rim_{id}": rim,
                f"rim_hull_{id}": hull,
            })

            lesion_stats.update({
                f"rim_voxels_{id}": rim_count,
                f"rim_volume_{id}": rim_count*voxel_volume,
            })

            lesion_stats.update({
                f"rim_hull_volume_{id}": hull.volume,
                f"rim_sphere_radius_{id}": rim_sphere_radius,
            })
        except Exception:
            print(prl_case['subid'], prl_case['lesion_index'])
            tb_str = traceback.format_exc()
            print(f"Captured Traceback:\n{tb_str}")
            pass

        # ---Process T2 Lesion---
        try:
            lesion = lesion_dx._get_center_lesion(index_crop, lab_data, index)
            lesion_count = int(lesion.sum())

            # get convex hull
            hull = lesion_dx.get_convex_hull(lesion, voxel_sizes=voxel_size)
            
            lesion_data.update({
                f"lesion_{id}": lesion,
                f"lesion_hull_{id}": hull,
            })

            lesion_stats.update({
                f"lesion_voxels_{id}": lesion_count,
                f"lesion_volume_{id}": lesion_count*voxel_volume,
            })
            lesion_stats.update({
                f"lesion_hull_volume_{id}": hull.volume
            })
        except Exception:
            tb_str = traceback.format_exc()
            print(f"Captured Traceback:\n{tb_str}")
            pass

        lesion_data[f"voxel_size_{id}"] = voxel_size    
    return lesion_stats, lesion_data

In [5]:
EXPERIMENT_KEYS

{'stage1': 'stage1_crop_lr_sweep',
 'stage2': 'stage2_numcrops_dicece',
 'stage3': 'stage3_numcrops_bkd_constwt115',
 'stage4': 'stage4_sweep_dicece_wts',
 'stage5': 'stage5_sweep_dicecewt_nbatch',
 'stage6': 'stage6_sweep_dicece_wts',
 'lambda_test': 'test_dicece_lambda'}

Define which experiment's model to use for inference

In [6]:
dataset_name = "roi_train2"
dataset = Dataset(dataset_name)

experiment_key, run_name = "stage6", "run1"
experiment_name = f"{EXPERIMENT_KEYS[experiment_key]}/{run_name}"
experiment_name = "sweep_dicecewts/run1"
experiment_name = "test_dicece_lambda/run1"
experiment = Experiment.from_run_dir(experiment_name, dataset)

expand_xy: int = experiment.preprocess_config.expand_xy
expand_z: int = experiment.preprocess_config.expand_z
images: tuple[str, ...] = experiment.preprocess_config.images

inference_dataset = Dataset("inference_dataset")
inference_dataset.create_datalist()

2026-05-30 19:33:22.967 | INFO     | core.dataset:create_datalist:205 - /home/srs-9/Projects/prl_project/training/inference_dataset/datalist_template.json exists; use rebuild=True to replace it


In [7]:
experiment.name

'test_dicece_lambda/run1'

In [8]:
dataset.cases

,,split,case_type,image,label,subject_dir
subid,lesion_index,,,,,
1293,1,testing,PRL,/media/smbshare/srs-9/prl_project/data/sub1293...,/media/smbshare/srs-9/prl_project/data/sub1293...,/media/smbshare/srs-9/prl_project/data/sub1293...
1074,3,testing,PRL,/media/smbshare/srs-9/prl_project/data/sub1074...,/media/smbshare/srs-9/prl_project/data/sub1074...,/media/smbshare/srs-9/prl_project/data/sub1074...
1080,3,testing,PRL,/media/smbshare/srs-9/prl_project/data/sub1080...,/media/smbshare/srs-9/prl_project/data/sub1080...,/media/smbshare/srs-9/prl_project/data/sub1080...
2131,1,testing,PRL,/media/smbshare/srs-9/prl_project/data/sub2131...,/media/smbshare/srs-9/prl_project/data/sub2131...,/media/smbshare/srs-9/prl_project/data/sub2131...
1011,7,testing,PRL,/media/smbshare/srs-9/prl_project/data/sub1011...,/media/smbshare/srs-9/prl_project/data/sub1011...,/media/smbshare/srs-9/prl_project/data/sub1011...
...,...,...,...,...,...,...
1076,15,fold3,Lesion,/media/smbshare/srs-9/prl_project/data/sub1076...,/media/smbshare/srs-9/prl_project/data/sub1076...,/media/smbshare/srs-9/prl_project/data/sub1076...
1050,99,fold4,Lesion,/media/smbshare/srs-9/prl_project/data/sub1050...,/media/smbshare/srs-9/prl_project/data/sub1050...,/media/smbshare/srs-9/prl_project/data/sub1050...
2131,6,fold0,Lesion,/media/smbshare/srs-9/prl_project/data/sub2131...,/media/smbshare/srs-9/prl_project/data/sub2131...,/media/smbshare/srs-9/prl_project/data/sub2131...


In [9]:
experiment.id.replace("/", "_")

'roi_train2_t1_sweep_dicecewts_run1'

In [10]:
inference_dataset = Dataset("inference_dataset")
inference_dataset.cases['inference'] = None
for i, case_i in tqdm(inference_dataset.cases.iterrows()):
    inference_dataset.cases.loc[i, "inference"] = find_inference(Path(case_i.subject_dir)/str(case_i.name[1]), experiment_name)
    
# dataset.cases['inference'] = None
# for i, case_i in tqdm(dataset.cases.iterrows()):
#     dataset.cases.loc[i, "inference"] = find_inference(Path(case_i.subject_dir)/str(case_i.name[1]), experiment_name)

# inference_dataset.cases.to_csv("tmp/full_inference_cases.csv")
# prl_df = inference_dataset.prl_df
# datalist_df = pd.DataFrame(inference_dataset.datalist_template['testing']).set_index(["subid", "lesion_index"])
# datalist_df.index.get_level_values("subid").unique()
# subid = 1082
# lesion_index = 10
# iloc = datalist_df.index.get_loc((subid, lesion_index))
# case_dict = datalist_df.reset_index().iloc[iloc].to_dict()

# inference_dataset.datalist_template['testing']
# test_case = inference_dataset.datalist_template['testing'][0]
# inference = get_infer_path(inference_dataset, test_case, experiment_name)
# images = inference_dataset.get_images(test_case, ["flair", "phase"])
# case_dict["inference"] = inference

# lst_lesion_index = inference_dataset.subject_dir(subid) / "lstai_lesion_index.nii.gz"
# labels = [inference, lst_lesion_index]
# for item in os.scandir(inference_dataset.data_root):
    

0it [00:00, ?it/s]

In [184]:
# from concurrent.futures import ProcessPoolExecutor

# def process_subject_func(subid):
#     res = lesion_dx.screen_for_iron(inference_dataset, subid, "label")
#     for l_idx, val in res.items():
         

# # Wrap your subject loop in this
# with ProcessPoolExecutor() as executor:
#     # Pass each subid to the screen_for_iron function
#     results = list(executor.map(process_subject_func, subids))

In [8]:
inference_dataset.cases

,,split,case_type,image,label,subject_dir,inference
subid,lesion_index,,,,,,
1126,4,testing,PRL,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...
1177,6,testing,PRL,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...
1152,9,testing,PRL,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...
2115,15,testing,PRL,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...
1126,1,testing,PRL,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...
...,...,...,...,...,...,...,...
1265,3,testing,Lesion,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...
1274,79,testing,Lesion,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...
1239,160,testing,Lesion,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...,/media/smbshare/srs-9/prl_project/inference_da...


In [ ]:
all_results = {}

for subid in tqdm(inference_dataset.cases.index.get_level_values("subid").unique()):
    # Dict of {lesion_index: bool}
    res = lesion_dx.screen_for_iron(inference_dataset, subid, "inference")
    # Store with the full (subid, lesion_index) tuple key
    for l_idx, val in res.items():
        all_results[(subid, l_idx)] = val

# Map the dictionary to the index to create the column all at once
inference_dataset.cases['has_iron'] = inference_dataset.cases.index.map(all_results).fillna(False)
# dataset.cases.to_csv("tmp/t1_training_cases.csv")


In [15]:
inference_dataset.cases['has_iron']

subid  lesion_index
1126   4                True
1177   6                True
1152   9                True
2115   15               True
1126   1                True
                       ...  
1265   3               False
1274   79              False
1239   160             False
1265   57              False
1177   27              False
Name: has_iron, Length: 3912, dtype: bool

In [8]:
cases = pd.read_csv("tmp/t1_training_cases.csv", index_col=["subid", "lesion_index"])
dataset.cases = cases
screened_cases = dataset.cases[dataset.cases['has_iron']]
screened_cases.groupby("subid")["has_iron"].sum()

subid
1010     5
1011     9
1033     2
1038     3
1044     7
1050     8
1074     3
1076    10
1078     7
1080     7
1124     6
1125     2
1131     1
1136     4
1215     5
1293     5
1298     3
1348     4
1358     2
1396     3
1529     7
2026     2
2041     7
2131     3
Name: has_iron, dtype: int64

In [ ]:
import analysis._cache as caching



In [13]:
import json
import subprocess

WORKER_SCRIPT = "/home/srs-9/Projects/prl_project/notebooks/radiomics_worker.py"
PYTHON_36 = "/home/srs-9/anaconda3/envs/pyradiomics2/bin/python"

cases = pd.read_csv("tmp/inference_cases.csv", index_col=["subid", "lesion_index"])
svname = "inference_pyradiomics.csv"
inference_dataset.cases = cases
screened_cases = inference_dataset.cases[inference_dataset.cases['has_iron']]
screened_cases.groupby("subid")["has_iron"].sum()

params = "/home/srs-9/Projects/prl_project/notebooks/tmp/radiomics_params.yaml"

all_features = []
for i in tqdm(range(len(screened_cases))):
    lesion_case = screened_cases.iloc[i]
    subid = lesion_case.name[0]
    lesion_index = lesion_case.name[1]
    lesion_dir = Path(lesion_case['image']).parent
    expand_suffix = f"_xy{inference_dataset.preprocess.expand_xy}_z{inference_dataset.preprocess.expand_z}"
    phase_path = lesion_dir / f"phase{expand_suffix}.nii.gz"
    inference_path = lesion_case['inference']

    input_data = {"image_path": str(phase_path), "mask_path": str(inference_path), "params": str(params)}
    with open("temp_in.json", "w") as f:
        json.dump(input_data, f)

    try:
        process = subprocess.run(
            [PYTHON_36, WORKER_SCRIPT, "temp_in.json", "temp_out.json"],
            capture_output=True,
            check=True,
            text=True
        )
    except subprocess.CalledProcessError as e:
        print(e.stdout)
        print(e.stderr)

    with open("temp_out.json", 'r') as f:
        features = json.load(f)
    features['subid'] = subid
    features['lesion_index'] = lesion_index
    all_features.append(features)
    
    
all_features_df = pd.DataFrame(all_features)
all_features_df = all_features_df.set_index(["subid", "lesion_index"])
all_features_df.to_csv(f"/home/srs-9/Projects/prl_project/notebooks/tmp/{svname}")

  0%|          | 0/442 [00:00<?, ?it/s]

In [9]:
process

NameError: name 'process' is not defined

In [25]:
full_data = defaultdict(list)
def analyze_wrapper(args):
    subid, lesion_indices = args
    return lesion_dx.analyze_subject_prl(subid, dataset, "inference", lesion_indices=lesion_indices)

tasks = []
for subid, g in screened_cases.reset_index().groupby("subid"):
    tasks.append((subid, g['lesion_index'].to_list()))
with BetterPool(8) as pool:
    results_iterator = pool.imap_unordered(analyze_wrapper, tasks)
    for result in tqdm(results_iterator, total=len(tasks)):
        full_data['prl_stats'].extend(result[0])
        full_data['prl_data'].extend(result[1])
tasks

  0%|          | 0/24 [00:00<?, ?it/s]

[(1010, [1, 4, 3, 7, 2]),
 (1011, [7, 16, 3, 12, 6, 4, 1, 5, 2]),
 (1033, [2, 1]),
 (1038, [2, 1, 3]),
 (1044, [2, 11, 8, 3, 4, 5, 21]),
 (1050, [1, 6, 4, 12, 3, 5, 14, 2]),
 (1074, [3, 1, 2]),
 (1076, [21, 2, 24, 3, 16, 7, 4, 12, 10, 1]),
 (1078, [10, 1, 9, 8, 2, 3, 5]),
 (1080, [3, 1, 10, 2, 4, 6, 9]),
 (1124, [1, 2, 11, 3, 4, 10]),
 (1125, [1, 3]),
 (1131, [1]),
 (1136, [5, 3, 1, 2]),
 (1215, [2, 3, 6, 5, 1]),
 (1293, [1, 2, 7, 6, 3]),
 (1298, [5, 1, 2]),
 (1348, [1, 2, 6, 3]),
 (1358, [1, 2]),
 (1396, [1, 3, 2]),
 (1529, [5, 11, 14, 2, 6, 7, 12]),
 (2026, [2, 1]),
 (2041, [1, 5, 3, 8, 4, 13, 2]),
 (2131, [1, 4, 2])]

In [26]:
analysis_dir = Path("/home/srs-9/Projects/prl_project/analysis/")
pd.DataFrame(full_data['prl_stats']).to_csv(
    analysis_dir/ f"prl_image_stats-{experiment.id.replace('/', '_')}.csv",
    index=False
)

In [ ]:
import pyperclip

cmd = open_itksnap_workspace_cmd(images, labels, rename_root=("/media/smbshare", "Z:/"))
pyperclip.copy(cmd)

In [11]:
cases = pd.read_csv("tmp/t1_inference_cases.csv", index_col=["subid", "lesion_index"])
dataset.cases = cases
screened_cases = dataset.cases[dataset.cases['has_iron']]
screened_cases.groupby("subid")["has_iron"].sum()

subid
1082     7
1101     7
1118     7
1126    19
1130     8
1133     9
1152     8
1156     5
1164    14
1165     3
1177     4
1178    11
1183    11
1186     8
1201    11
1209     2
1234     7
1239    31
1248     3
1252    13
1257     8
1262     5
1265    17
1274     9
1281    15
1282     9
1292     7
1296    10
1316    10
1327     8
1353     6
1376     4
1395    10
1453    12
1497    13
1508     2
1523     9
1537     5
1546     7
2003     5
2011     2
2017     6
2060    12
2086    14
2087     7
2095    14
2098     9
2115    11
2122     3
2135     1
2168    13
2207     7
Name: has_iron, dtype: int64

In [12]:
cases = pd.read_csv("tmp/inference_cases.csv", index_col=["subid", "lesion_index"])
dataset.cases = cases
screened_cases = dataset.cases[dataset.cases['has_iron']]
screened_cases.groupby("subid")["has_iron"].sum()

subid
1082     6
1101     7
1118     5
1126    18
1130     7
1133     9
1152     8
1156     5
1164    12
1165     3
1177     4
1178    10
1183    10
1186     9
1201    11
1209     1
1234     7
1239    36
1248     4
1252    12
1257     8
1262     3
1265    16
1274     9
1281    16
1282     7
1292     6
1296    10
1316    10
1327     7
1353     5
1376     4
1395    11
1453    15
1497    11
1508     3
1523     9
1537     5
1546     7
2003     4
2011     2
2017     4
2060    11
2086    15
2087     6
2095    12
2098     8
2115    12
2122     3
2135     1
2168    11
2207     7
Name: has_iron, dtype: int64